# Nepali Emotional Text-to-Speech with Indic Parler-TTS

This notebook generates Nepali speech in different emotional tones using the
**base** `ai4bharat/indic-parler-tts` model.

**Speaker:** `Amrita` is the only real, supported Nepali speaker for this model.
Any other name (`Srijana`, `Sagar`, `Sita`, etc.) is out-of-vocabulary. An
out-of-vocabulary speaker name leaves the caption unconstrained in voice
space, so generation drifts off the Nepali speaker manifold. In an earlier
round of this work, the out-of-vocabulary name "Srijana" scored better on an
automatic Speech Emotion Recognition (SER) metric than "Amrita" (56.2% mean
vs. 40.6%), but it sounded non-native and low quality to every human
listener. **A better SER score from an out-of-vocabulary speaker name is an
artifact, not a real result.** Always confirm a speaker name is actually
supported by the model, and always confirm by ear, not just by metric.

**Recommended configuration (this notebook uses the base model, not a
finetuned checkpoint):**

| Emotion | SER top-1 (Amrita) | Status |
|---|---|---|
| neutral | 100% | good |
| happy | 75% | best result of the sweep |
| angry | 50% | cleanest separation, little leakage into happy |
| sad | 0% | unsolved, see the metric caveat above |

Full experiment details are in `CAPTIONS.txt`.


## 1. Install dependencies

Installs the `parler-tts` package and makes sure `protobuf` is new enough
for the tokenizers used below.

In [ ]:
%%capture
!pip install parler-tts
!pip install "protobuf>=5.28.0" --upgrade


## 2. Load the model and tokenizers

Two important, easy-to-get-wrong details:

1. **Use the base model deliberately.** `model_id = "ai4bharat/indic-parler-tts"`
   is the base model, not a finetuned checkpoint. Do not swap in a different
   checkpoint here.
2. **Two tokenizers are required.** The description string is encoded by the
   model's text encoder (`flan-t5-large`), which has a different vocabulary
   from the repo's own prompt tokenizer. Using one tokenizer for both fails
   silently: a caption like "Amrita speaks in a sad tone." reaches the model
   as garbled, meaningless subwords, and all caption control (emotion,
   speaker, pace) is lost with no error raised. Building
   `description_tokenizer` from `model.config.text_encoder._name_or_path` is
   what makes caption control work at all.

In [ ]:
import torch
import soundfile as sf
from transformers import AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration

# The BASE model, deliberately -- not a finetuned checkpoint.
MODEL_ID = "ai4bharat/indic-parler-tts"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)

# The prompt (spoken text) tokenizer belongs to the repo itself.
prompt_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# The description (caption) tokenizer belongs to the model's text encoder.
# This is the single easiest thing to get wrong: reusing prompt_tokenizer
# here will not error, it will just silently destroy caption control.
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)

SR = model.config.sampling_rate
print("description tokenizer:", model.config.text_encoder._name_or_path, "| sampling rate:", SR)


## 3. Generate emotional speech (Amrita, four emotions)

The text prompts below are deliberately emotion-neutral: any emotion heard
in the output comes from the description caption, not from the words
themselves. This keeps the test a fair check of caption-driven control.

Emotion is requested through pace, energy, and pitch contour in the
caption, never through voice quality words, since the speaker's inherent
voice quality should stay constant across emotions. `"Amrita"` in the
caption is the speaker token itself: dropping it lets the voice drift
between runs.

In [ ]:
# Emotion-neutral sentences: any emotion you hear came from the caption, not the words.
TEXTS = [
    "आजको बैठक बिहान दस बजे सभाकक्षमा सुरु हुनेछ।",
    "उहाँ हिजो साँझ काठमाडौंबाट फर्कनुभयो।",
    "यो बाटो सिधै बजारसम्म पुग्छ।",
]

QUALITY = " The recording is very high quality, clear and close-sounding, with no background noise."
EMOTIONS = {
    "neutral": "Amrita speaks in a neutral tone at a moderate pace with balanced pitch.",
    "happy":   "Amrita speaks in a happy, cheerful tone, expressive and lively at a slightly fast pace.",
    "sad":     "Amrita speaks in a sad, sorrowful tone, subdued and slow, her delivery trailing downward.",
    "angry":   "Amrita's angry tone, with a high pitch voice, is captured with exceptional quality in a close-sounding environment.",
}

def synth(text, description, seed=0):
    """Generate one audio clip for a given spoken text and caption."""
    torch.manual_seed(seed)
    d = description_tokenizer(description + QUALITY, return_tensors="pt").to(DEVICE)
    p = prompt_tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        gen = model.generate(
            input_ids=d.input_ids, attention_mask=d.attention_mask,
            prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask,
            do_sample=True, temperature=0.8,
            max_new_tokens=1500,  # guard: a broken caption can babble for 30s of audio
        )
    return gen.to(torch.float32).cpu().numpy().squeeze()

clips = {}
for emo, desc in EMOTIONS.items():
    for i, t in enumerate(TEXTS):
        clips[(emo, i)] = synth(t, desc, seed=1234 + i)
        print(f"  {emo:8s} sent{i}  {len(clips[(emo, i)]) / SR:5.2f}s")


### Save and play back each sentence across all emotions

Clips are normalized against one shared divisor rather than per clip: a
per-clip normalization would erase the loudness difference between, say,
angry and sad, and that loudness difference is part of the emotion itself.

The block below prints and plays each sentence once for every emotion, so
you can compare the same words spoken four different ways.

In [ ]:
import numpy as np
import IPython.display as ipd

# ONE shared divisor: per-clip normalization would delete the loudness
# difference between angry and sad, which is part of the emotion itself.
scale = max(np.abs(a).max() for a in clips.values()) or 1.0

for i in range(len(TEXTS)):
    print(f"\n{'=' * 60}\nSENTENCE {i}: {TEXTS[i]}\n{'=' * 60}")
    for emo in EMOTIONS:
        a = clips[(emo, i)] / scale
        sf.write(f"{emo}_{i}.wav", a, SR)
        print(f"[{emo.upper()}]")
        ipd.display(ipd.Audio(a, rate=SR, autoplay=False))


## 4. Objective separation check

Do not trust your ears alone. Since the same three sentences were generated
under all four emotion captions, MFCC features and pitch (F0) can be
compared across emotions to check whether the classes actually separate in
a measurable way. A `separation` value greater than 1 means the emotions are
more different from each other (between-class) than they are from
themselves across sentences (within-class).

Note the caveat from the top of this notebook: this kind of automatic score
can be fooled by an out-of-vocabulary speaker name, so a good score here is
not a substitute for listening to the clips above.

In [ ]:
import librosa

EM = list(EMOTIONS)
feat = {
    e: np.array([
        librosa.feature.mfcc(y=clips[(e, i)] / scale, sr=SR, n_mfcc=13).mean(1)
        for i in range(len(TEXTS))
    ])
    for e in EM
}

for e in EM:
    f0 = [
        np.nanmedian(librosa.pyin(clips[(e, i)] / scale, fmin=70, fmax=500, sr=SR)[0])
        for i in range(len(TEXTS))
    ]
    print(f"{e:8s} F0 median {np.nanmedian(f0):6.1f} Hz")

between = [
    np.linalg.norm(feat[a].mean(0) - feat[b].mean(0))
    for idx, a in enumerate(EM) for b in EM[idx + 1:]
]
within = [np.mean([np.linalg.norm(x - feat[e].mean(0)) for x in feat[e]]) for e in EM]
print(f"\nseparation = {np.mean(between) / np.mean(within):.2f}  (greater than 1 is separable)")


## 5. Broad multi-sentence check (single steady caption)

A separate, wider test: one steady, formal caption applied across many
different Nepali sentences, to check general voice quality and
intelligibility outside of the emotion sweep above. Output is printed and
played for every sentence in the list.

In [ ]:
TRAINING_DESCRIPTIONS = [
    "Amrita speaks with a deep, formal Nepali voice. Her speech is clear, steady and authoritative with natural pacing in a quiet noise-free environment.",
]

sentences = [
    "नमस्ते, तपाईंलाई आज कस्तो सहयोग चाहिन्छ?",
    "म तपाईंको सहायक बोल्दैछु।",
    "कृपया आफ्नो समस्या विस्तारमा बताइदिनुहोस्।",
    "आजको मौसम निकै राम्रो देखिन्छ।",
    "तपाईंको दिन शुभ रहोस्।",
    "म नेपाली भाषामा पनि कुरा गर्न सक्छु।",
    "के तपाईंलाई कुनै जानकारी चाहिएको छ?",
    "तपाईंले पठाएको अनुरोध प्रक्रिया हुँदैछ।",
    "कृपया केही क्षण प्रतीक्षा गर्नुहोस्।",
    "धन्यवाद, तपाईंको सन्देश प्राप्त भयो।",
    "यो एउटा परीक्षण वाक्य हो।",
    "कम्प्युटर विज्ञान निकै रोचक विषय हो।",
    "म नयाँ प्रविधिहरू सिक्दैछु।",
    "नेपाल प्राकृतिक सौन्दर्यले भरिएको देश हो।",
    "काठमाडौं नेपालको राजधानी शहर हो।",
    "आज तपाईंले के सिक्नुभयो?",
    "संगीत सुन्न मलाई मन पर्छ।",
    "कृत्रिम बुद्धिमत्ता भविष्यको महत्वपूर्ण प्रविधि हो।",
    "तपाईंको इन्टरनेट जडान स्थिर देखिन्छ।",
    "कृपया फेरि प्रयास गर्नुहोस्।",
    "यो आवाज परीक्षणको लागि प्रयोग गरिएको वाक्य हो।",
    "विद्यालयमा विद्यार्थीहरू अध्ययन गर्दैछन्।",
    "हामी नयाँ परियोजनामा काम गरिरहेका छौं।",
    "तपाईंको फाइल सफलतापूर्वक अपलोड भयो।",
    "अब म अर्को वाक्य पढ्दैछु।",
    "तपाईंलाई सहयोग गर्न पाउँदा खुशी लाग्यो।",
    "सुरक्षित यात्रा गर्नुहोस्।",
    "तपाईंको अर्डर तयार हुँदैछ।",
    "कृपया आफ्नो नाम भन्नुहोस्।",
    "यो प्रणाली अहिले सक्रिय अवस्थामा छ।",
    "न्छन् जसमा १४ वटा स्वरवर्ण र ३३ वटा व्यञ्जनवर्ण ",
    "ुन्छन्। यो संसारमा सबैभन्दा बढी प्रयोग हुने लिपि मध्ये चौथो हो जहा यस लिपिमा १२० भन्दा बढी भाषाहरू लेखिन्छन्।",
]

# Encode the description once outside the loop since it does not change.
desc_inputs = description_tokenizer(TRAINING_DESCRIPTIONS[0], return_tensors="pt").to(DEVICE)

for i, sentence in enumerate(sentences):
    prompt_inputs = prompt_tokenizer(sentence, return_tensors="pt").to(DEVICE)

    with torch.inference_mode():
        gen = model.generate(
            input_ids=desc_inputs.input_ids,
            attention_mask=desc_inputs.attention_mask,
            prompt_input_ids=prompt_inputs.input_ids,
            prompt_attention_mask=prompt_inputs.attention_mask,
            max_new_tokens=1000,
        )

    audio = gen.cpu().numpy().squeeze().astype(np.float32)
    max_val = np.abs(audio).max()
    if max_val > 1e-6:
        audio = audio / max_val

    print(f"\n[Sentence {i + 1}] {sentence}")
    ipd.display(ipd.Audio(audio, rate=SR))
